# Entrenar el detector de placas con YOLO11

Reemplaza el `best.pt` actual (YOLOv8n entrenado a 640 px sin rotacion) por un YOLO11 entrenado
para corregir las **dos debilidades que se midieron con la app en produccion**:

| Debilidad medida | Causa | Que cambia aqui |
|---|---|---|
| Una foto rotada 90 grados da **cero detecciones** | se entreno con `degrees=0.0`: nunca vio una placa inclinada | `degrees=15`, `perspective`, `shear` |
| Pierde placas lejanas (**3 de 5** en la escena de trafico) | `yolov8n` (3.0M parametros) a 640 px | `yolo11s`, `scale=0.6` |

> **Sobre TensorFlow:** Ultralytics YOLO11 corre sobre **PyTorch**, no TensorFlow. Transfer
> learning si: se parte de pesos preentrenados en COCO. Si algun dia hace falta un `.tflite` para
> movil, se exporta al final con `model.export(format="tflite")`, pero el entrenamiento es PyTorch.

> **Por que YOLO11 y no YOLO12:** el servidor corre `ultralytics 8.4.154`, que carga YOLO11 sin
> problema. Y YOLO12 usa capas de atencion **mas lentas en CPU**, que es donde corre la inferencia
> de este proyecto (`t3.micro`, sin GPU). YOLO11 es el punto correcto para este despliegue.

## 1. Comprobar la GPU

Colab: *Entorno de ejecucion* -> *Cambiar tipo de entorno* -> **T4 GPU**.
Sin GPU esto tardaria dias en vez de ~1 hora.

In [ ]:
!nvidia-smi

import torch
print("CUDA disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NINGUNA - cambia el entorno a T4")

## 2. Instalar

In [ ]:
!pip install -q ultralytics roboflow

import ultralytics
ultralytics.checks()

## 3. Descargar el dataset

Dataset elegido: **License Plate Recognition**, de Roboflow Universe Projects
(`roboflow-universe-projects/license-plate-recognition-rxg4e`).

Por que este y no uno colombiano:

- **10.125 imagenes**, 722 estrellas y 37.9k descargas: es el mas contrastado del tema.
- Exporta directo a formato **YOLOv11**, licencia CC BY 4.0.
- El dataset `placas-colombianas` (1.770 imagenes colombianas) **no tiene ninguna version
  generada**, asi que no se puede descargar por API. Los colombianos descargables son muy
  pequenos (~100 imagenes).

Detectar el *rectangulo* de la placa depende poco del pais: la forma es la misma. Lo especifico de
Colombia (placa amarilla, formato AAA123) vive en el OCR y en la correccion de formato del
servidor, no en el detector. En la seccion 7 hay un ajuste fino opcional con datos colombianos.

**Necesitas tu API key de Roboflow:** entra a roboflow.com, cuenta gratis, *Settings -> API Keys*.
Pegala abajo y **no la subas al repositorio**.

In [ ]:
from roboflow import Roboflow

API_KEY = ""  # <-- pega aqui tu API key de Roboflow

rf = Roboflow(api_key=API_KEY)
proyecto = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")

# Se listan las versiones para elegir con criterio en vez de a ciegas
for v in proyecto.versions():
    print("version", v.version_number, ":", v.name)

Al descargar, **ojo con la resolucion**: varias versiones vienen *estiradas a
640x640*, y entrenar a 960 px sobre imagenes de 640 no anade detalle real. La celda siguiente
descarga e **informa del tamano real** para que la decision sea con datos.

In [ ]:
VERSION = 4   # cambiala si el listado de arriba muestra una sin resize a 640

dataset = proyecto.version(VERSION).download("yolov11")
print("\nDescargado en:", dataset.location)

# --- Mirar que hay de verdad dentro, sin asumir nada ---
import glob
from collections import Counter

import yaml
from PIL import Image

with open(dataset.location + "/data.yaml") as fh:
    cfg = yaml.safe_load(fh)
print("\nclases:", cfg["names"], "| nc:", cfg.get("nc"))

for split in ("train", "valid", "test"):
    n = len(glob.glob(dataset.location + "/" + split + "/images/*"))
    print(split + ":", n, "imagenes")

muestras = glob.glob(dataset.location + "/train/images/*")[:200]
tamanos = Counter(Image.open(p).size for p in muestras)
print("\ntamanos mas comunes (200 muestras):", tamanos.most_common(3))
print("^ si todo es (640, 640), entrena con imgsz=640: subirlo no anade informacion")

## 4. Entrenar

Cada hiperparametro responde a algo medido en produccion, no es una receta copiada:

| Parametro | Valor | Por que |
|---|---|---|
| `model` | `yolo11s.pt` | 9.4M parametros frente a los 3.0M del `yolov8n` actual: el salto que mas rinde en objetos pequenos sin volverse lento en CPU |
| `degrees` | **15.0** | el fallo mas grave del modelo actual: con `degrees=0.0`, una foto rotada da cero detecciones |
| `scale` | 0.6 | ensena a ver la placa a distancias muy distintas; las lejanas son las que se pierden |
| `perspective` / `shear` | 0.0005 / 3.0 | las placas se fotografian de lado, no de frente |
| `fliplr` | **0.0** | una placa espejada no existe; el texto invertido solo confunde |
| `hsv_s` / `hsv_v` | 0.7 / 0.4 | placas al sol, en sombra y de noche |
| `close_mosaic` | 10 | las ultimas epocas ven imagenes reales completas, no mosaicos |
| `patience` | 25 | corta si deja de mejorar; el entrenamiento original tenia 100 y nunca cortaba |

### Cuanto cuesta `yolo11s` en el servidor

El modelo entrenado tiene que correr en una `t3.micro` **sin GPU**. Medido en esa instancia
(mediana de 3 inferencias, con warmup previo):

| modelo | parametros | 640 | 960 | 1280 |
|---|---|---|---|---|
| `yolov8n` (el actual) | 3.0M | 0.08 s | 0.15 s | 0.30 s |
| `yolo11n` | 2.6M | 0.08 s | 0.16 s | 0.35 s |
| **`yolo11s`** | 9.5M | 0.17 s | 0.42 s | **0.78 s** |

El servidor infiere a 1280, asi que `yolo11s` anade ~0.5 s por foto sobre el modelo actual.
Dentro de los 2.6-5.3 s que tarda una peticion completa es asumible, y se paga por detectar las
placas lejanas que hoy se pierden.

**Si la latencia molesta**, entrena `yolo11n` en vez de `yolo11s`: cuesta lo mismo que el modelo
actual y aun asi gana la rotacion y las aumentaciones, que es el fallo mas grave.

In [ ]:
from ultralytics import YOLO

IMGSZ = 640   # subelo a 960 SOLO si la celda anterior mostro imagenes mayores que 640

modelo = YOLO("yolo11s.pt")   # transfer learning desde los pesos de COCO

resultados = modelo.train(
    data=dataset.location + "/data.yaml",
    epochs=100,
    imgsz=IMGSZ,
    batch=16,
    name="placas_yolo11",
    patience=25,
    seed=0,

    # --- rotacion y geometria: el fallo principal del modelo actual ---
    degrees=15.0,
    perspective=0.0005,
    shear=3.0,
    scale=0.6,

    # --- color e iluminacion ---
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    # --- una placa espejada no existe ---
    fliplr=0.0,
    flipud=0.0,

    mosaic=1.0,
    close_mosaic=10,
)

PESOS = str(resultados.save_dir) + "/weights/best.pt"
print("\nPesos entrenados:", PESOS)

## 5. Validar

`mAP50-95` es la metrica estandar, pero **el numero que manda es el del banco real** (seccion 6):
un mAP bonito sobre imagenes de internet no garantiza leer mejor las placas de la demo.

In [ ]:
metricas = YOLO(PESOS).val(data=dataset.location + "/data.yaml", imgsz=IMGSZ)
print("mAP50    :", round(float(metricas.box.map50), 4))
print("mAP50-95 :", round(float(metricas.box.map), 4))
print("precision:", round(float(metricas.box.mp), 4))
print("recall   :", round(float(metricas.box.mr), 4), " <- el recall es lo que importa en placas lejanas")

## 6. La prueba que decide: rotacion y placas del banco

El modelo actual falla con fotos rotadas. Esta celda lo comprueba **directamente**, girando la
imagen en los cuatro sentidos. El modelo viejo daba:

```
0 grados       -> 1 caja
90 horario     -> 0 cajas    <-- el fallo
180            -> 1 caja
90 antihorario -> 0 cajas    <-- el fallo
```

Sube una o varias fotos de la carpeta `pruebas/` del repositorio y ejecuta:

In [ ]:
import cv2
from google.colab import files

print("Sube fotos de la carpeta pruebas/ del repositorio:")
subidas = files.upload()

modelo_nuevo = YOLO(PESOS)
giros = [
    ("0 grados", None),
    ("90 horario", cv2.ROTATE_90_CLOCKWISE),
    ("180", cv2.ROTATE_180),
    ("90 antihorario", cv2.ROTATE_90_COUNTERCLOCKWISE),
]

for nombre in subidas:
    img = cv2.imread(nombre)
    print("\n###", nombre)
    for etiqueta, giro in giros:
        f = img if giro is None else cv2.rotate(img, giro)
        r = modelo_nuevo.predict(source=f, conf=0.25, imgsz=1280, verbose=False)[0]
        confs = [round(float(c), 2) for c in r.boxes.conf.cpu().numpy()]
        print("  " + etiqueta.ljust(16), len(r.boxes), "cajas  confs=", confs)

Para dar el entrenamiento por bueno:

1. **Las cuatro orientaciones detectan la placa.** Si `90 horario` sigue dando 0 cajas, la
   rotacion no se aprendio: sube `degrees` a 25 y reentrena.
2. **Mas cajas en la escena de trafico.** El modelo actual encuentra 3 de 5 placas en
   `COH262-IJO387-WCT308-FRL260-VCU458_trafico.jpg`. Si el nuevo encuentra 4 o 5, ganaste.

## 7. (Opcional) Ajuste fino con placas colombianas

Un ajuste fino corto sobre datos colombianos puede ayudar con el amarillo y el formato propio.
Son pocas imagenes, asi que van **pocas epocas y learning rate bajo**: con mas se sobreajusta y
se pierde lo aprendido de las 10.000.

In [ ]:
# Verificado: itm-mprof/placas-colombia tiene v2 (~100 imagenes)
proy_co = rf.workspace("itm-mprof").project("placas-colombia")
for v in proy_co.versions():
    print("version", v.version_number, ":", v.name)

ds_co = proy_co.version(2).download("yolov11")

with open(ds_co.location + "/data.yaml") as fh:
    print("\nclases colombianas:", yaml.safe_load(fh)["names"])
print("^ si los nombres de clase NO coinciden con el dataset grande, hay que igualarlos antes")

In [ ]:
# Ajuste fino: pocas epocas, lr bajo, partiendo de los pesos ya entrenados
ajustado = YOLO(PESOS).train(
    data=ds_co.location + "/data.yaml",
    epochs=20,
    imgsz=IMGSZ,
    batch=8,
    lr0=0.0005,        # 20x mas bajo que el 0.01 por defecto
    degrees=15.0,
    fliplr=0.0,
    scale=0.5,
    name="placas_yolo11_co",
)

PESOS_CO = str(ajustado.save_dir) + "/weights/best.pt"
print("Pesos ajustados:", PESOS_CO)
print("Compara ESTE con el de la seccion 6 antes de quedartelo: con ~100 imagenes es facil empeorar")

## 8. Descargar y desplegar

**Se conserva el modelo anterior.** Si el nuevo no supera las 10 lecturas correctas de 13 del
banco, se vuelve atras.

In [ ]:
from google.colab import files

files.download(PESOS)   # o PESOS_CO si el ajuste fino resulto mejor

```bash
# 1. Subir el modelo nuevo SIN borrar el que funciona
scp -i llaveVype.pem best.pt ubuntu@34.225.169.137:/home/ubuntu/proyecto/best_v2.pt

# 2. Cambiarlo, guardando el anterior
ssh -i llaveVype.pem ubuntu@34.225.169.137 "cd /home/ubuntu/proyecto && \
  cp best.pt best_v1.pt && cp best_v2.pt best.pt && sudo systemctl restart yolo-plates"

# 3. Medir con el mismo banco de siempre
python pruebas/probar_api.py --url http://34.225.169.137:8080

# 4. Si NO supera 10/13, volver atras
ssh -i llaveVype.pem ubuntu@34.225.169.137 "cd /home/ubuntu/proyecto && \
  cp best_v1.pt best.pt && sudo systemctl restart yolo-plates"
```

> **Lo que este entrenamiento NO arregla:** que el OCR confunda `M` con `H` en una placa borrosa
> (`SMV098` se lee `SHV098`). Eso es el reconocedor de texto, no el detector. La via para eso es
> entrenar un YOLO11 de **caracteres** (36 clases: A-Z y 0-9) sobre recortes de placa, y sustituir
> a EasyOCR/PaddleOCR. Candidato a inspeccionar:
> `reconocimiento-de-placas-vehiculares/reconocimiento_de_placas` (1.071 imagenes, v1, exporta
> YOLOv11). Hay que bajarlo y mirar su `data.yaml` para confirmar si sus clases son caracteres
> individuales antes de contar con el.